# Object Detection with Faster R-CNN

Faster R-CNN is a powerful object detection model used by developers across industries for tasks like surveillance, autonomous driving, and image analysis. In this example, you'll detect objects in images using a pre-trained Faster R-CNN model. The model identifies objects like people, cars, animals, and furniture, drawing bounding boxes around each detection with confidence scores.

The model is pre-trained on the COCO dataset, which contains 80 common object categories including person, car, dog, cat, bicycle, and many more. It uses a two-stage detection process: first identifying potential object regions, then classifying what each object is. The model is approximately 160MB and will be downloaded from PyTorch on first run.

The code below installs the necessary packages. TorchVision provides the pre-trained Faster R-CNN model and detection utilities:

In [ ]:
!pip install -q torch torchvision pillow matplotlib numpy

In the code below, we import the required libraries. TorchVision handles the model and image transformations, while Matplotlib will be used for visualizing the detection results:

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import urllib.request
from io import BytesIO
import requests
import os

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device == "cpu":
    print("⚠️ Warning: Running on CPU. Detection will be slower.")
    print("   For best results, use a GPU-enabled resource.")

The code below loads the pre-trained Faster R-CNN model with a ResNet-50 backbone. The model will be downloaded automatically on first run and cached for future use. We set the model to evaluation mode since we're only doing inference, not training:

In [ ]:
print("Loading Faster R-CNN model...")
print("(First run may take a moment to download the model)\n")

# Load pre-trained model with updated weights API
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()
model = model.to(device)

print("✓ Model loaded successfully!")

The COCO dataset contains 80 object categories. In the code below, we define the class names that correspond to the model's predictions:

In [ ]:
# COCO class names (91 total, but only 80 are used)
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

print(f"Model can detect {len([c for c in COCO_CLASSES if c != 'N/A'])} object categories")
print("\nSample categories:", COCO_CLASSES[1:11])

## Object Detection

The code below defines our detection function. It takes an image, runs it through the model, and returns the detected objects with their locations (bounding boxes), class labels, and confidence scores. We filter out low-confidence detections using a threshold:

In [ ]:
def detect_objects(image, model, threshold=0.5):
    """
    Detect objects in an image.

    Args:
        image: PIL Image or numpy array
        model: Faster R-CNN model
        threshold: Confidence threshold (0-1)

    Returns:
        boxes: Bounding box coordinates [x1, y1, x2, y2]
        labels: Class IDs for each detection
        scores: Confidence scores for each detection
    """
    # Convert to PIL Image if needed
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)

    # Transform image to tensor
    transform = transforms.Compose([transforms.ToTensor()])
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Run detection
    with torch.no_grad():
        predictions = model(image_tensor)

    # Extract predictions
    pred = predictions[0]

    # Filter by threshold
    mask = pred['scores'] > threshold
    boxes = pred['boxes'][mask].cpu().numpy()
    labels = pred['labels'][mask].cpu().numpy()
    scores = pred['scores'][mask].cpu().numpy()

    return boxes, labels, scores

Now let's create a function to visualize the detections. The code below draws bounding boxes around detected objects, adds labels with confidence scores, and uses different colors for better visibility:

In [ ]:
def visualize_detections(image, boxes, labels, scores, class_names, threshold=0.5):
    """
    Draw bounding boxes on image with labels and scores.

    Args:
        image: PIL Image or numpy array
        boxes: Array of bounding boxes [x1, y1, x2, y2]
        labels: Array of class IDs
        scores: Array of confidence scores
        class_names: List of class names
        threshold: Only show detections above this confidence
    """
    # Convert to numpy if PIL Image
    if isinstance(image, Image.Image):
        image = np.array(image)

    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)

    # Color palette for different object classes
    colors = plt.cm.hsv(np.linspace(0, 1, len(class_names)))

    for box, label, score in zip(boxes, labels, scores):
        if score < threshold:
            continue

        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1

        # Select color based on class
        color = colors[label % len(colors)]

        # Draw rectangle
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)

        # Add label with background
        label_text = f"{class_names[label]}: {score:.2f}"
        ax.text(
            x1, y1 - 5, label_text,
            bbox=dict(facecolor=color, alpha=0.7, edgecolor='none', pad=2),
            color='white', fontsize=10, weight='bold'
        )

    ax.axis('off')
    plt.tight_layout()
    return fig

## Test with Sample Images

Let's test our object detection on some sample images. The code below downloads a few test images from the web and runs detection on them. You'll see bounding boxes appear around detected objects!

In [ ]:
# Sample image URLs
sample_images = [
    "https://images.unsplash.com/photo-1543466835-00a7907e9de1",  # Dog
    "https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d",  # Person
    "https://images.unsplash.com/photo-1549399542-7e3f8b79c341",  # car
    "https://images.pexels.com/photos/1133957/pexels-photo-1133957.jpeg",  # Bird
]

print("Downloading and processing sample images...\n")

for idx, url in enumerate(sample_images):
    try:
        print(f"Processing image {idx + 1}/{len(sample_images)}...")

        # Use requests with headers
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Open image from memory
        image = Image.open(BytesIO(response.content)).convert("RGB")

        # Detect objects
        boxes, labels, scores = detect_objects(image, model, threshold=0.5)

        print(f"  Found {len(boxes)} objects")

        # Show detected object types
        detected_classes = [COCO_CLASSES[label] for label in labels]
        print(f"  Detected: {', '.join(detected_classes)}\n")

        # Visualize
        fig = visualize_detections(image, boxes, labels, scores, COCO_CLASSES)
        plt.show()

    except Exception as e:
        print(f"  Error processing image {url}: {e}\n")

print("✓ Sample detection complete!")

## Upload Your Own Images

Now you can try the detector on your own images! The code below allows you to upload an image using path directory or URL of the image and see what objects are detected:

In [ ]:
def detect_in_uploaded_image(image_input, threshold=0.5):
    """
    Detect objects in an uploaded image from either local path or URL.

    Args:
        image_input: Can be either:
                    - Local file path (e.g., '/content/image.jpg')
                    - Image URL (e.g., 'https://example.com/image.jpg')
        threshold: Confidence threshold for detection (0.0 to 1.0)

    Returns:
        boxes, labels, scores: Detection results
    """
    try:
        # Check if input is a URL
        if image_input.startswith(('http://', 'https://')):
            print(f"📡 Downloading image from URL: {image_input}")

            # Download image from URL with headers
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
            response = requests.get(image_input, headers=headers, timeout=10)
            response.raise_for_status()

            # Open image from memory
            image = Image.open(BytesIO(response.content)).convert("RGB")
            print("✅ Image downloaded successfully")

        else:
            # Treat as local file path
            print(f"📁 Loading image from path: {image_input}")

            # Check if file exists
            if not os.path.exists(image_input):
                raise FileNotFoundError(f"Image file not found: {image_input}")

            image = Image.open(image_input).convert("RGB")
            print("✅ Image loaded successfully")

        # Display original image
        print(f"📊 Image size: {image.size}")

        # Detect objects
        boxes, labels, scores = detect_objects(image, model, threshold=threshold)

        # Display results
        print(f"\n🎯 Detected {len(boxes)} objects (threshold: {threshold}):")
        for i, (label, score) in enumerate(zip(labels, scores)):
            print(f"  {i+1}. {COCO_CLASSES[label]}: {score:.1%} confidence")

        # Visualize detections
        fig = visualize_detections(image, boxes, labels, scores, COCO_CLASSES, threshold)
        plt.show()

        return boxes, labels, scores

    except Exception as e:
        print(f"❌ Error: {e}")
        return None, None, None



This code create an interactive options to use the code above, you can add the image path or URL link for classification

In [ ]:
def interactive_detection():
    """
    Interactive version that asks for input
    """
    print("🤖 Object Detection - Enter your image source")
    print("=" * 50)

    # Get image source
    image_source = input("Enter image path or URL: ").strip()

    # Get threshold with default value
    threshold_input = input("Confidence threshold (0.1-1.0) [default: 0.5]: ").strip()
    threshold = float(threshold_input) if threshold_input else 0.5

    print("\n" + "=" * 50)

    # Run detection
    return detect_in_uploaded_image(image_source, threshold)

You can classify your image here by providing the image path or URL. Let's test with this link incase you dont have an image ready "https://plus.unsplash.com/premium_photo-1667030489905-d8e6309ebe0e?ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1pbi1zYW1lLXNlcmllc3wxfHx8ZW58MHx8fHx8&auto=format&fit=crop&q=60&w=200"

In [ ]:
interactive_detection()

## Conclusion

We have now set up [Faster R-CNN]("https://github.com/trzy/FasterRCNN") for object detection on a GPU! The model can detect 80 different object types in images, drawing bounding boxes around each detection with confidence scores. Once you've got this working, you have all the tools you need to build object detection applications on Saturn Cloud.

If you wanted to process images at scale or work with video streams, you could distribute the workload across multiple GPUs using distributed computing - this can be achieved using [saturn cloud](https://app.community.saturnenterprise.io/auth/hosted-registration?_gl=1*vubx0h*_gcl_au*NDg0MjEzNzI5LjE3NTk3OTIwOTY.*_ga*MTMzMjE2NDcxMi4xNzU5NzkyMDk2*_ga_9QKGCS5Q41*czE3NjA0Njk4MjkkbzckZzEkdDE3NjA0NzE4MDMkajU5JGwwJGgw). You could also fine-tune the model on custom datasets to detect specialized objects specific to your use case.